# Kaggriculture Playground

This notebook is a safe place to learn how a Kaggriculture agent works before submitting anything to Kaggle. We will run the game on this computer, make small changes, and see what happens.

Each section below explains one step. Add a Python code cell immediately below it as you work.

In [1]:
print("Hello world!!!")

Hello world!!!


## 1. Import the game simulator

The `kaggle_environments` package contains a local copy of the Kaggriculture game engine. Import the function that creates a game.

Add a code cell below this one to import the simulator.

In [2]:
# kaggle_environments is the package that contains the local game simulator.
# "make" creates one game environment for us to run.
from kaggle_environments import make

## 2. Choose the two players

A match always has two agents. Start with Kaggle's built-in `random` agent for both players. Later, replace the first player with your own `main.py` file.

Add a code cell below this one to store the agent and opponent choices in variables.

In [3]:
agent = "random"
opponent = "random"

## 3. Create a game

Create a Kaggriculture environment and choose how many turns it should last. There are 24 turns in one in-game day and 720 turns in a complete 30-day season.

Begin with 24 turns so that each experiment runs quickly. Add a code cell below this one to create the game.

In [4]:
steps = 24

env = make(
    "kaggriculture",
    configuration={"episodeSteps": steps},
    debug=True,
)

## 4. Run the match

Tell the environment to run the two players. On every turn, the game sends each player an observation, receives an action dictionary back, applies both actions, and advances the simulation.

Add a code cell below this one to run the match.

In [5]:
# perf_counter is a stopwatch designed for timing short pieces of code.
from time import perf_counter

start_time = perf_counter()
run_steps = env.run([agent, opponent])
elapsed_seconds = perf_counter() - start_time

print(f"Match completed: {len(run_steps)} turns in {elapsed_seconds:.2f} seconds")

final_step = run_steps[-1]
for player, state in enumerate(final_step):
    print(
        f"Player {player}: "
        f"reward={state.reward}, status={state.status}"
    )

Match completed: 24 turns in 0.10 seconds
Player 0: reward=2950.0, status=DONE
Player 1: reward=2750.0, status=DONE


## 5. Read the final result

After the match, the environment keeps a history of every turn. The final entry contains each player's final reward and status. A normal match should finish with a `DONE` status.

Add a code cell below this one to print the final result for both players.

In [6]:
for player, state in enumerate(env.steps[-1]):
    print(f"Player {player}: reward={state.reward}, status={state.status}")

Player 0: reward=2950.0, status=DONE
Player 1: reward=2750.0, status=DONE


## 6. Read a short match timeline

The game records every turn, but most turns are just walking around. The next cell follows player 0: it prints their farmer's grid position each turn, the action they requested, and any money change. Positions use `(x, y)`: x increases to the east (right) and y increases to the south (down).

Run this cell after the match has finished.

In [7]:
# We are following player 0 only.
player = 0
previous_money = None

# Look through the saved game one turn at a time.
for turn, turn_states in enumerate(env.steps):
    state = turn_states[player]
    observation = state.observation
    farm = observation["farms"][player]
    x, y = farm["farmer"]
    money = farm["money"]

    # This is the action player 0 requested for this turn.
    action = state.action or {}
    farmer_action = action.get("farmer", ["PASS"])
    action_text = " ".join(map(str, farmer_action))

    money_text = ""
    if previous_money is not None and money != previous_money:
        money_text = f" | money: ${previous_money:.0f} → ${money:.0f}"

    print(
        f"Turn {turn:>3} | day {observation['day'] + 1:>2}, hour {observation['hour'] + 1:>2} "
        f"| position ({x}, {y}) | action: {action_text}{money_text}"
    )
    previous_money = money

Turn   0 | day  1, hour  1 | position (4, 4) | action: PASS
Turn   1 | day  1, hour  2 | position (5, 4) | action: EAST | money: $3000 → $2950
Turn   2 | day  1, hour  3 | position (5, 4) | action: HARVEST
Turn   3 | day  1, hour  4 | position (5, 4) | action: HARVEST
Turn   4 | day  1, hour  5 | position (5, 4) | action: PLANT TOMATO
Turn   5 | day  1, hour  6 | position (5, 3) | action: NORTH
Turn   6 | day  1, hour  7 | position (5, 3) | action: WATER
Turn   7 | day  1, hour  8 | position (4, 3) | action: WEST
Turn   8 | day  1, hour  9 | position (4, 3) | action: PLANT TOMATO
Turn   9 | day  1, hour 10 | position (4, 3) | action: PASS
Turn  10 | day  1, hour 11 | position (3, 3) | action: WEST
Turn  11 | day  1, hour 12 | position (3, 2) | action: NORTH
Turn  12 | day  1, hour 13 | position (3, 2) | action: PASS
Turn  13 | day  1, hour 14 | position (4, 2) | action: EAST
Turn  14 | day  1, hour 15 | position (3, 2) | action: WEST
Turn  15 | day  1, hour 16 | position (3, 1) | actio

In [8]:
print("hello!")

hello!


## 7. Try your own agent

The carrot policies are Python modules. Import their `agent` functions directly rather than passing their file paths to the local Kaggle loader. Direct imports preserve the package context required by their shared implementation.

Run this notebook from the repository root so the `agents` package is importable. The two names below are ordinary Python callables: the simulator calls each one once per turn with its observation.

In [9]:
# The conveyor policy always tries to grow carrots. It is our simple baseline.
from agents.carrot.conveyor import agent as conveyor_agent

# The decision policy uses the explicit carrot belief/utility model.
from agents.carrot.decision import agent as decision_agent

### Run a reproducible carrot matchup

Use the same seed and configuration when comparing policies. A fixed seed does not make two different policies follow the same actions; their actions still change the market and therefore their individual game trajectories. It does make the initial random setup and random-event stream repeatable for a given pairing.

This cell runs the decision policy as player 0 against the conveyor baseline as player 1. Both agents receive their own observations and act in the same shared game. Use a full 720-turn season to see the crop cycle and market effects.

In [10]:
# Keep this seed unchanged when comparing a baseline and one candidate.
seed = 42

# Kaggriculture's default full season: 24 turns/day × 30 days.
steps = 720

env = make(
    "kaggriculture",
    configuration={"episodeSteps": steps, "seed": seed},
    debug=True,
)
# The list is in player order: decision policy is player 0; conveyor is player 1.
env.run([decision_agent, conveyor_agent])

for player, state in enumerate(env.steps[-1]):
    label = "decision" if player == 0 else "conveyor"
    print(f"Player {player} ({label}): reward={state.reward}, status={state.status}")

Player 0 (decision): reward=8849.0, status=DONE
Player 1 (conveyor): reward=8729.0, status=DONE


### Repeat with the player positions swapped

Player position can matter in a shared-market game. Repeat the same seed with the policies reversed before interpreting a result. A small initial suite might use several seeds, each run twice: once with the decision policy as player 0 and once as player 1.

Do not change a belief parameter because of one result. First collect a fixed set of matched runs, then compare the final rewards, crop outcomes, and—when we add traces—the calibration of the policy's probability estimates.

In [11]:
# Reuse the same seed and number of turns from the previous cell.
swapped_env = make(
    "kaggriculture",
    configuration={"episodeSteps": steps, "seed": seed},
    debug=True,
)

# Now conveyor is player 0 and the decision policy is player 1.
swapped_env.run([conveyor_agent, decision_agent])

for player, state in enumerate(swapped_env.steps[-1]):
    label = "conveyor" if player == 0 else "decision"
    print(f"Player {player} ({label}): reward={state.reward}, status={state.status}")

Player 0 (conveyor): reward=8789.0, status=DONE
Player 1 (decision): reward=8809.0, status=DONE


## 8. Run the matched evaluation suites

A useful comparison runs each seed twice: once with the decision policy as player 0 and once as player 1. The same seed makes the randomized setup repeatable for that particular pairing; reversing seats checks whether player order influenced the result.

The `pass` opponent is only a health check, so it uses ten seeds. The other opponents use the fixed 30-seed suite `42` through `71`. Run one cell at a time and keep its printed summary with the source revision and parameter values.

For timing work, use the separate `experiments/memoization/profile_step8.py` script rather than these evaluation cells. It measures simulator time and agent time separately, writes a CSV and a chart, and keeps profiling overhead out of the timing measurements. Add `--record-traces` on a live baseline run to save decision observations/actions; use `--replay-traces` afterward to reuse the recorded outcome only when the current agent produces exactly the same action at every turn.

In [12]:
# This helper runs both player orders for every predeclared seed.
# Keeping the seed list fixed prevents us from selecting only favorable games after seeing results.
from statistics import mean

def run_matched_suite(opponent, opponent_name, seeds):
    results = []

    # Run the same matchup twice per seed so player position is balanced.
    for seed in seeds:
        for players, decision_player in (
            ([decision_agent, opponent], 0),
            ([opponent, decision_agent], 1),
        ):
            env = make(
                'kaggriculture',
                configuration={'episodeSteps': 720, 'seed': seed},
                debug=True,
            )
            env.run(players)

            # The final state contains each player's season reward and status.
            final_states = env.steps[-1]
            opponent_player = 1 - decision_player
            decision_reward = final_states[decision_player].reward
            opponent_reward = final_states[opponent_player].reward
            results.append({
                'seed': seed,
                'decision_player': decision_player,
                'decision_reward': decision_reward,
                'opponent_reward': opponent_reward,
                'difference': decision_reward - opponent_reward,
                'done': all(state.status == 'DONE' for state in final_states),
            })

    differences = [row['difference'] for row in results]
    wins = sum(difference > 0 for difference in differences)
    completed = sum(row['done'] for row in results)
    print(f'decision vs {opponent_name}: {len(results)} matches')
    print(f'mean decision advantage: {mean(differences):.1f} coins')
    print(f'decision wins: {wins}/{len(results)}; normal completions: {completed}/{len(results)}')
    return results

### Health check: decision policy versus `pass`

The `pass` agent takes no useful action. This is not a serious strength comparison; it verifies that the decision policy can complete a season and carry out the basic carrot lifecycle. Ten matched seeds make this regression check inexpensive.

In [13]:
# Use ten fixed seeds for the inexpensive health-check suite.
pass_results = run_matched_suite('pass', 'pass', range(42, 52))

decision vs pass: 20 matches
mean decision advantage: 4642.1 coins
decision wins: 20/20; normal completions: 20/20


### Robustness check: decision policy versus `random`

The built-in `random` agent supplies legal but unspecialized play. This checks that the decision policy remains effective when the other farm creates irregular, unpredictable market activity.

In [14]:
# Use the fixed 30-seed evaluation suite and both player positions.
random_results = run_matched_suite('random', 'random', range(42, 72))

decision vs random: 60 matches
mean decision advantage: 8671.3 coins
decision wins: 60/60; normal completions: 60/60


### Reference check: decision policy versus `starter`

The built-in `starter` agent is a deterministic one-tile carrot loop. It sells carrots, keeps a seed on hand, and repeatedly plants, waters, and harvests. It is a stronger reference than `random`, while still being easy to understand.

In [15]:
# Keep the same seed suite used for the other substantive comparisons.
starter_results = run_matched_suite('starter', 'starter', range(42, 72))

decision vs starter: 60 matches
mean decision advantage: 5104.4 coins
decision wins: 60/60; normal completions: 60/60


### Primary comparison: decision policy versus carrot conveyor

The conveyor policy is the current simple baseline: it continuously grows carrots without the plant-versus-pass probability model. This is the central test of whether the decision network earns its added complexity. The first ten runs already completed for seeds `42` through `46` belong to this 60-match cell if the source revision and parameter values have not changed.

In [16]:
# This is the primary economic comparison: belief-based policy against the simple baseline.
conveyor_results = run_matched_suite(conveyor_agent, 'carrot conveyor', range(42, 72))

decision vs carrot conveyor: 60 matches
mean decision advantage: 77.3 coins
decision wins: 55/60; normal completions: 60/60


### Historical comparison: decision policy versus `test_agent_v0`

`test_agent_v0` is an earlier standalone, multi-crop symbolic/probabilistic agent. It is a historical reference rather than a policy to optimize against, but it helps reveal whether the deliberately narrow carrot model is competitive with the broader earlier design.

In [17]:
# Import the standalone historical agent directly from the archive.
from agents.archive.test_agent_v0 import agent as test_agent_v0_agent

# Evaluate it with the same seeds and seat swaps as the other substantive opponents.
test_agent_v0_results = run_matched_suite(
    test_agent_v0_agent, 'historical test_agent_v0', range(42, 72)
)

decision vs historical test_agent_v0: 60 matches
mean decision advantage: 5369.8 coins
decision wins: 60/60; normal completions: 60/60


### Historical `main_v0` note

The archived `main_v0.py` is a partial source fragment: it defines helper functions but has no `agent(obs)` entry point. It is retained as historical material, not included in the runnable suite. Restore or wrap a verified entry point before treating it as an opponent.

These five runnable suites contain 260 matches: 20 for `pass` and 60 each for `random`, `starter`, carrot conveyor, and `test_agent_v0`. Record the resulting rows before considering parameter changes.

## 9. What to learn from each run

Do not worry about winning immediately. First check that your agent finishes every game, returns valid actions, and does not let crops die or inventory overflow. Then make one small change at a time and compare the results across several games.